# 常用评测框架介绍
本次作业我们主要介绍常用的几种大型语言模型评测框架：
## 框架概览
| 框架名称 | 开发机构 | 主要特点 | 适用场景 |
|---------|---------|---------|---------|
| [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) | EleutherAI | 功能丰富，支持多种模型和任务，学术界标准 | 学术研究、基准测试 |
| [evalscope](https://github.com/modelscope/evalscope) | ModelScope | 支持自定义数据集组合，可视化分析，中文友好 | 产业应用、模型评估 |
| [Evalchemy](https://github.com/mlfoundations/evalchemy) | ML Foundations | 轻量级，注重可复现性和扩展性 | 研究实验、快速原型 |
| [lighteval](https://github.com/huggingface/lighteval) | Hugging Face | 集成Transformers生态，易于使用 | Hugging Face用户 |
我们将重点介绍**lm-evaluation-harness**和**evalscope**的基本使用方法。 

# 1. lm-evaluation-harness

我们介绍第一种常用的评测框架 `lm-evaluation-harness`，lm-eval要从 GitHub 仓库安装软件包，请在命令行运行：

```bash
git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
cd lm-evaluation-harness
pip install -e .
pip install -e .[math]

```

我们首先介绍它的简单使用，这里我们通过一个小模型 gpt2 来测试一下，请在命令行运行如下代码：

```bash
lm_eval --model hf \
    --model_args pretrained=openai-community/gpt2 \
    --tasks hellaswag \
    --limit 10 \
    --device cuda:0 \
    --batch_size 8
```

大概等待不到一分钟的时间，我们即可得到结果：

|  Tasks  |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|---------|------:|------|-----:|--------|---|----:|---|-----:|
|hellaswag|      1|none  |     0|acc     |↑  |  0.3|±  |0.1528|
|         |       |none  |     0|acc_norm|↑  |  0.3|±  |0.1528|

接下来我们介绍一种更复杂的使用方法，它可以对模型进行多维的能力评测，我们将其划分为通用语言理解、常识推理、代码、数学推理四类任务。

| 维度        | 任务集合                                               |
| --------- | -------------------------------------------------- |
| 通用（Gen）   | LAMBADA(LAMBADA Standard、LAMBADA OpenAI), TriviaQA |
| 常识推理（Com） | PIQA, ARC-easy                                     |
| 代码（Code）  | HumanEval, MBPP                                    |
| 数学（Math）  | GSM8K, Minerva Math                                |

In [4]:
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# eval_model_path = "/home/magnus-share/xuhu/model/Qwen2___5-Math-1___5B"
eval_model_path = "openai-community/gpt2"
zero_shot_tasks = ["arc_easy", "piqa", "lambada", "triviaqa"]
few_shot_tasks = ["humaneval", "mbpp", "gsm8k", "minerva_math"]
all_results = {}

# 🔥 关键：只创建一次模型实例
lm = HFLM(
    pretrained=eval_model_path,      # 可以是路径，会自动加载
    tokenizer=eval_model_path,
    batch_size=32,
    device="cpu",
    max_length=1024,
)

# ===== 1. Zero-shot evaluation =====
print(f"Evaluating zero-shot tasks: {zero_shot_tasks}")
results_zero = evaluator.simple_evaluate(
    model=lm,  # ← 复用同一个 lm 实例
    tasks=zero_shot_tasks,
    num_fewshot=0,
    limit=1,
    batch_size=32,
    gen_kwargs={"max_gen_toks": 512}, 
    # device='cuda',
    confirm_run_unsafe_code=True
)
all_results.update(results_zero['results'])

# ===== 2. Few-shot evaluation =====
print(f"Evaluating 3-shot tasks: {few_shot_tasks}")
results_few = evaluator.simple_evaluate(
    model=lm,  # ← 同一个 lm 实例
    tasks=few_shot_tasks,
    num_fewshot=3,
    limit=1,
    batch_size=32,
    gen_kwargs={"max_gen_toks": 512}, 
    # device='cuda',
    confirm_run_unsafe_code=True
)
all_results.update(results_few['results'])

print("Combined results keys:", list(all_results.keys()))
print("all_results:",all_results)


# os.makedirs(
#     f'results/eval_result',
#     exist_ok=True
# )
# save_filepath = os.path.join(
#     f'results/eval_result',
#     f"multi-dimension-eval.json"
# )
# with open(save_filepath, "w") as file:
#     json.dump(all_results, file, indent=4)

# print(f"Combined evaluation results saved to {save_filepath}")


generation_kwargs: {'max_gen_toks': 512} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!


Evaluating zero-shot tasks: ['arc_easy', 'piqa', 'lambada', 'triviaqa']


Running generate_until requests:   9%|▉         | 576/6477 [16:36<2:50:13,  1.73s/it]
Overwriting default num_fewshot of triviaqa from None to 0
Overwriting default num_fewshot of lambada_openai from None to 0
Overwriting default num_fewshot of lambada_standard from None to 0
Overwriting default num_fewshot of piqa from None to 0
Overwriting default num_fewshot of arc_easy from None to 0
Running loglikelihood requests: 100%|██████████| 8/8 [00:00<00:00, 32.08it/s]
generation_kwargs: {'max_gen_toks': 512} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!


Evaluating 3-shot tasks: ['humaneval', 'mbpp', 'gsm8k', 'minerva_math']


Using the latest cached version of the module from /Users/xuhu.6736/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--code_eval/78d307ea938083398db7d9815f03ed661e9c15f60d77880ce007a8a02648f176 (last modified on Mon Jan 19 21:27:53 2026) since it couldn't be found locally at evaluate-metric--code_eval, or remotely on the Hugging Face Hub.
Overwriting default num_fewshot of minerva_math_algebra from 4 to 3
Overwriting default num_fewshot of minerva_math_counting_and_prob from 4 to 3
Overwriting default num_fewshot of minerva_math_geometry from 4 to 3
Overwriting default num_fewshot of minerva_math_intermediate_algebra from 4 to 3
Overwriting default num_fewshot of minerva_math_num_theory from 4 to 3
Overwriting default num_fewshot of minerva_math_prealgebra from 4 to 3
Overwriting default num_fewshot of minerva_math_precalc from 4 to 3
Overwriting default num_fewshot of gsm8k from 5 to 3
Overwriting default num_fewshot of mbpp from 3 to 3
100%|██████████| 1/1 [00:00<00

Combined results keys: ['arc_easy', 'lambada_openai', 'lambada_standard', 'piqa', 'triviaqa', 'gsm8k', 'humaneval', 'mbpp', 'minerva_math', 'minerva_math_algebra', 'minerva_math_counting_and_prob', 'minerva_math_geometry', 'minerva_math_intermediate_algebra', 'minerva_math_num_theory', 'minerva_math_prealgebra', 'minerva_math_precalc']
all_results: {'arc_easy': {'alias': 'arc_easy', 'acc,none': 0.0, 'acc_stderr,none': 'N/A', 'acc_norm,none': 1.0, 'acc_norm_stderr,none': 'N/A'}, 'lambada_openai': {'alias': 'lambada_openai', 'perplexity,none': 48895.074567592026, 'perplexity_stderr,none': 'N/A', 'acc,none': 0.0, 'acc_stderr,none': 'N/A'}, 'lambada_standard': {'alias': 'lambada_standard', 'perplexity,none': 26270.121698137566, 'perplexity_stderr,none': 'N/A', 'acc,none': 0.0, 'acc_stderr,none': 'N/A'}, 'piqa': {'alias': 'piqa', 'acc,none': 1.0, 'acc_stderr,none': 'N/A', 'acc_norm,none': 1.0, 'acc_norm_stderr,none': 'N/A'}, 'triviaqa': {'alias': 'triviaqa', 'exact_match,remove_whitespace':

# 2. evalscoep

## 2.1 环境安装

In [ ]:
# 创建conda环境 (可选)

# 建议使用 python 3.10
conda create -n evalscope python=3.10

# 激活conda环境
conda activate evalscope

# pip安装依赖
pip install evalscope
pip install 'evalscope[app]' -U  # 可视化依赖包


## 2.2 第一步：定义 Schema

我们需要创建一个 CollectionSchema，在其中列出选用的数据集及其权重。

> 提示：weight 可以是任意正数，EvalScope 会自动进行归一化处理。通过 flatten() 方法可以预览最终每个数据集的实际占比。

In [1]:
from evalscope.collections import CollectionSchema, DatasetInfo

# 嵌套结构示例：数学组 + 推理组
schema = CollectionSchema(name='reasoning_index', datasets=[
    CollectionSchema(name='math', weight=0.5, datasets=[
        DatasetInfo(name='gsm8k', weight=0.5, tags=['en']),
        DatasetInfo(name='aime25', weight=0.5, tags=['en']),
    ]),
    CollectionSchema(name='logic', weight=0.5, datasets=[
        DatasetInfo(name='arc', weight=0.5, tags=['en']),
        DatasetInfo(name='ceval', weight=0.5, tags=['zh'], args={'subset_list': ['logic']}),
    ]),
])

# 打印查看归一化后的权重分布
print(schema.flatten())


/opt/miniconda3/envs/evalscope/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DatasetInfo(name='gsm8k', weight=0.25, task_type='', tags=['en'], args={}, hierarchy=['reasoning_index', 'math']), DatasetInfo(name='aime25', weight=0.25, task_type='', tags=['en'], args={}, hierarchy=['reasoning_index', 'math']), DatasetInfo(name='arc', weight=0.25, task_type='', tags=['en'], args={}, hierarchy=['reasoning_index', 'logic']), DatasetInfo(name='ceval', weight=0.25, task_type='', tags=['zh'], args={'subset_list': ['logic']}, hierarchy=['reasoning_index', 'logic'])]


## 2.3 第二步：采样数据

定义好 Schema 后，我们需要按照策略抽取样本。EvalScope 提供了多种采样器：

- 加权采样：样本数量与你设置的权重成正比。权重越高的任务，在测试集中出现的题目越多，对总分的影响也越大。这最能体现“业务导向”。

- StratifiedSampler（分层采样）：保持原数据集的样本规模比例（适合不做人为干预的客观统计）。

- UniformSampler（均匀采样）：所有数据集样本数相同（适合横向对比各能力短板）。

针对“构建指数”场景，我们推荐使用 加权采样 (WeightedSampler)。


In [2]:
from evalscope.collections import WeightedSampler
from evalscope.utils.io_utils import dump_jsonl_data

# 初始化加权采样器
sampler = WeightedSampler(schema)

# 采样 100 条数据作为最终测试集
# 根据权重，知识问答 30 条，长文本检索 30 条，指令遵循 40 条
# 实际采样数量可根据需要调整
mixed_data = sampler.sample(count=10)

# 将混合好的数据保存为 JSONL 文件，这就是你的“指数评测集”
dump_jsonl_data(mixed_data, 'data/index_testset.jsonl')

Sampling data:   0%|          | 0/4 [00:00<?, ?it/s]2026-01-20 11:59:44 - evalscope - INFO: No model is provided, using DummyCustomModel for testing.
2026-01-20 11:59:51 - evalscope - INFO: Loading dataset AI-ModelScope/gsm8k from modelscope > subset: main > split: test ...
Processing records: 100%|██████████| 1319/1319 [00:00<00:00, 313888.62it/s]
2026-01-20 12:00:03 - evalscope - INFO: Loading dataset AI-ModelScope/gsm8k from modelscope > subset: main > split: train ...
Sampling data:  25%|██▌       | 1/4 [00:23<01:10, 23.62s/it]2026-01-20 12:00:07 - evalscope - INFO: No model is provided, using DummyCustomModel for testing.
2026-01-20 12:00:07 - evalscope - INFO: Loading dataset opencompass/AIME2025 from modelscope > subset: AIME2025-I > split: test ...
Processing records: 100%|██████████| 15/15 [00:00<00:00, 86659.17it/s]
2026-01-20 12:00:16 - evalscope - INFO: Loading dataset opencompass/AIME2025 from modelscope > subset: AIME2025-II > split: test ...
Sampling data:  50%|█████    

## 2.4 第三步：统一评测

现在，你拥有了一个名为 rag_index_testset.jsonl 的文件。在 EvalScope 看来，它就是一个普通的本地数据集。我们直接调用 run_task 即可。



In [1]:
import os
from evalscope import TaskConfig, run_task

task_cfg = TaskConfig(
    # model='qwen2.5-14b-instruct', # 评测模型
    # # 使用一个提供 OpenAI 兼容接口的模型进行评测
    # # 可以是云上的API，也可以是通过vllm等框架部署的本地模型
    # api_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
    # api_key=os.getenv('DASHSCOPE_API_KEY'),
    # eval_type='openai_api',
    model='openai-community/gpt2', # 评测模型
    
    # 关键配置：指定数据集为 'data_collection' 模式
    datasets=['data_collection'],
    dataset_args={
        'data_collection': {
            'local_path': 'data/index_testset.jsonl', # 指向刚才生成的文件
            'shuffle': True # 打乱顺序
        }
    },
    eval_batch_size=5, # 根据你的 API 并发限额调整
    generation_config={
        'temperature': 0.0 # 评测通常设为 0 以保证结果可复现
    },
    use_cache="outputs/20260119_232050", # 复用本地缓存路径的推理结果和评测结果
)

run_task(task_cfg=task_cfg)

/opt/miniconda3/envs/evalscope/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-20 13:03:57 - evalscope - INFO: No eval_type is provided, setting eval_type to CHECKPOINT.
2026-01-20 13:03:57 - evalscope - INFO: Args: Task config is provided with TaskConfig type.
2026-01-20 13:03:57 - evalscope - INFO: Set resume from outputs/20260119_232050
2026-01-20 13:03:57 - evalscope - INFO: Running with native backend
2026-01-20 13:03:57 - evalscope - INFO: Dump task config to outputs/20260119_232050/configs/task_config_b42323.yaml
2026-01-20 13:03:57 - evalscope - INFO: {
    "model": "openai-community/gpt2",
    "model_id": "gpt2",
    "model_args": {
        "revision": "master",
        "precision": "torch.float16"
    },
    "model_task": "text_generation",
    "chat_template": null,
    "datasets": [


`torch_dtype` is deprecated! Use `dtype` instead!


                                                                       
                                                                       2026-01-20 13:04:02 - evalscope - INFO: Model loaded successfully.
Evaluating [data_collection]:   0%|          | 0/1 [00:04<?, ?subset/s]Token indices sequence length is longer than the specified maximum sequence length for this model (4885 > 1024). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (1024). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
                                                                       
                                       

{'data_collection': Report(name='data_collection', dataset_name='data_collection', dataset_pretty_name='', dataset_description='', model_name='gpt2', score=0.0, metrics=[Metric(name='acc', num=10, score=0.0, macro_score=0.0, categories=[Category(name=('reasoning_index', 'logic'), num=6, score=0.0, macro_score=0.0, subsets=[Subset(name='arc/ARC-Easy', score=0.0, num=2), Subset(name='ceval/logic', score=0.0, num=4)]), Category(name=('reasoning_index', 'math'), num=4, score=0.0, macro_score=0.0, subsets=[Subset(name='aime25/AIME2025-I', score=0.0, num=1), Subset(name='aime25/AIME2025-II', score=0.0, num=1), Subset(name='gsm8k/main', score=0.0, num=2)])])], analysis='N/A')}

评测完成后，你会在日志中看到 Overall report table，包含在每个自己上的评测结果：

+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| Model   | Dataset       | Metric   | Subset             |   Num |   Score | Cat.0           | Cat.1   |  
+=========+===============+==========+====================+=======+=========+=================+=========+    
| gpt2    | index_testset | Average  | arc/ARC-Challenge  |     1 |       0 | reasoning_index | logic   |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | arc/ARC-Easy       |     1 |       0 | reasoning_index | logic   |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | ceval/logic        |     4 |       0 | reasoning_index | logic   |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | aime25/AIME2025-I  |     1 |       0 | reasoning_index | math    |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | aime25/AIME2025-II |     1 |       0 | reasoning_index | math    |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | gsm8k/main         |     2 |       0 | reasoning_index | math    |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+  
| gpt2    | index_testset | Average  | OVERALL            |    10 |       0 | -               |         |  
+---------+---------------+----------+--------------------+-------+---------+-----------------+---------+   

## 2.5 第四步：Case分析

拿到 Index 总分并不代表工作结束。相反，分析模型为什么取得这个分数通常更有价值：是模型知识储备不足？还是仅仅因为没按格式输出被误判？

EvalScope 内置了可视化分析工具，让你非常方便查看每一个样本的详细情况。无需编写额外代码，只需在终端运行一条命令，即可启动本地服务：


In [ ]:
evalscope app

接下来访问 `http://localhost:7860` 或者 `http://0.0.0.0:7860` 即可看到可视化界面。
  
<center align="center">
    <img src="images/evalscope_panel.png" width="800"/>
</center>

# 3. Evalchemy

## 3.1 简介

[Evalchemy](https://github.com/mlfoundations/evalchemy) 是一个轻量级的评测框架，由ML Foundations开发。它注重可复现性、扩展性和简洁性，特别适合研究人员进行快速实验和原型开发。

## 3.2 主要特点

- **轻量级设计**：核心代码简洁，易于理解和修改
- **高度可扩展**：支持自定义任务、指标和模型
- **可复现性**：内置实验跟踪和版本控制
- **研究导向**：适合学术研究和快速原型开发

## 3.3 安装和使用

```bash
pip install evalchemy
```

基本使用示例：

```python
import evalchemy as ec

# 定义一个简单的评测任务
@ec.task
def simple_qa(model, question):
    prompt = f"Question: {question}\nAnswer:"
    response = model.generate(prompt)
    return response

# 创建评测配置
config = ec.Config(
    model="gpt2",
    tasks=["simple_qa"],
    metrics=["exact_match", "f1_score"]
)

# 运行评测
results = ec.run(config)
```

# 4. LightEval

## 4.1 简介

[LightEval](https://github.com/huggingface/lighteval) 是Hugging Face开发的评测框架，深度集成Transformers生态系统。它提供了统一的接口来评测Hugging Face模型，支持多种任务类型和评测指标。

## 4.2 主要特点

- **Transformers集成**：无缝对接Hugging Face模型和数据集
- **任务多样性**：支持文本生成、问答、分类等多种任务
- **分布式评测**：支持多GPU并行评测
- **易于使用**：简洁的API设计

## 4.3 安装和使用

```bash
pip install lighteval
```

基本使用示例：

```python
from lighteval import LightEval
from transformers import AutoModelForCausalLM, AutoTokenizer

# 加载模型和tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# 创建评测器
evaluator = LightEval(
    model=model,
    tokenizer=tokenizer,
    tasks=["hellaswag", "winogrande"],  # 要评测的任务
    num_fewshot=0,  # few-shot示例数量
    batch_size=8,
    max_length=512
)

# 运行评测
results = evaluator.run()
print(results)
```

## 4.4 命令行使用

LightEval也支持命令行使用：

```bash
lighteval evaluate \
    --model "gpt2" \
    --tasks "hellaswag,winogrande" \
    --num_fewshot 0 \
    --batch_size 8
```

# 5. 总结

这四个评测框架各有特色：

- **lm-evaluation-harness**：学术界标准，功能最全面
- **evalscope**：产业应用导向，支持自定义数据集组合和可视化
- **Evalchemy**：轻量级，适合研究和快速原型
- **LightEval**：Hugging Face生态集成，易于使用

选择哪个框架主要取决于你的具体需求：
- 如果进行学术研究，推荐使用lm-evaluation-harness
- 如果需要构建自定义评测集和可视化分析，推荐evalscope
- 如果是Hugging Face用户，推荐LightEval
- 如果需要快速原型和实验，推荐Evalchemy